# CN x1.0 — Canonical baseline model notebook

**Model ID:** `cn_x1_0`  
**Role:** first named CN selected-pool research baseline  
**Status:** `baseline_research_data_review`; `trade_ready=false`

This notebook is the human-readable companion to `configs/models/cn_x1_0.yaml`. It records the immutable model contract, complete snapshot-bound backtest evidence, reproduction commands, provider-drift warning and version-upgrade rules.

## 1. What CN x1.0 is

CN x1.0 ranks the governed CN130 equity pool against CSI 300 using XGBoost `rank:ndcg`. PR #344 evaluated eight pre-registered variants and retained the existing balanced OHLCV configuration. The model contract is canonical; the historical performance is not snapshot independent.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import yaml

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook inside the Alpha Engine repository.")

ROOT = find_repo_root()

MODEL_ID = "cn_x1_0"
CONFIG_PATH = ROOT / "configs/models/cn_x1_0.yaml"
REGISTRY_PATH = ROOT / "configs/models/model_registry_v1.yaml"

config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
registry = yaml.safe_load(REGISTRY_PATH.read_text(encoding="utf-8"))

assert config["model_id"] == MODEL_ID
assert config["display_name"] == "CN x1.0"
assert config["trade_ready"] is False
assert registry["models"][MODEL_ID]["config"] == str(CONFIG_PATH.relative_to(ROOT))
config["display_name"], config["status"], config["provider_binding"]["drift_review_issue"]

## 2. Universe and data contract

- Universe: `cn_selected_equities_v3`, 130 declared equities.
- Benchmark: CSI 300 (`000300`), reference only.
- Membership: static curated, with explicit survivorship-bias disclosure.
- Canonical evidence provider identity: `bf5fa1373a0b5ebfedcd90c2cf3c4748300efd2b25da0adfbfb1daab8c6405d8`.
- Provider cutoff: 2026-07-31.
- Issue #345 is a hard gate because another promotion-eligible snapshot produced materially different historical candidate returns.

In [ ]:
universe = config["universe"]
provider = config["provider_binding"]
pd.DataFrame(
    [
        {"field": "universe_id", "value": universe["universe_id"]},
        {"field": "declared_candidate_count", "value": universe["declared_candidate_count"]},
        {"field": "benchmark", "value": config["benchmark"]},
        {"field": "canonical_provider_identity", "value": provider["canonical_evidence_provider_identity_sha256"]},
        {"field": "prior_provider_identity", "value": provider["prior_provider_identity_sha256"]},
        {"field": "provider_cutoff", "value": provider["cutoff"]},
        {"field": "drift_review_issue", "value": provider["drift_review_issue"]},
    ]
)

## 3. Features, target and model parameters

`cn_balanced_ohlcv` combines momentum, reversal, volatility/range and liquidity features. The training target is a daily cross-sectional percentile rank converted into five gain levels. Economic evaluation uses raw 10-session forward returns.

In [ ]:
pd.DataFrame(
    {"expression": config["features"]["expressions"]}
).rename_axis("feature_index").reset_index()

In [ ]:
model = config["model"]
label = config["label"]
strategy = config["strategy"]

pd.DataFrame(
    [
        {"parameter": "family", "value": model["family"]},
        {"parameter": "objective", "value": model["objective"]},
        {"parameter": "gain_bins", "value": label["gain_bins"]},
        {"parameter": "num_boost_round", "value": model["num_boost_round"]},
        {"parameter": "max_leaves", "value": model["max_leaves"]},
        {"parameter": "min_data_in_leaf", "value": model["min_data_in_leaf"]},
        {"parameter": "learning_rate", "value": model["learning_rate"]},
        {"parameter": "seed", "value": model["seed"]},
        {"parameter": "holding_sessions", "value": strategy["holding_sessions"]},
        {"parameter": "rebalance_sessions", "value": strategy["rebalance_sessions"]},
        {"parameter": "top_n", "value": strategy["top_n"]},
        {"parameter": "weighting", "value": strategy["weighting"]},
        {"parameter": "cost_bps", "value": strategy["cost_bps"]},
    ]
)

## 4. Complete development backtest

The authoritative compounded relative excess formula is:

`(1 + compounded_strategy_return) / (1 + compounded_benchmark_return) - 1`

The canonical evidence below is bound to provider identity `bf5fa...c6405d8`.

In [ ]:
development = config["backtest_evidence"]["development"]
calculated_relative = (
    (1.0 + development["compounded_strategy_return"])
    / (1.0 + development["compounded_benchmark_return"])
    - 1.0
)
assert abs(calculated_relative - development["compounded_relative_excess_return"]) < 1e-10

pd.DataFrame(
    [
        {"metric": "compounded_strategy_return", "value": development["compounded_strategy_return"]},
        {"metric": "compounded_benchmark_return", "value": development["compounded_benchmark_return"]},
        {"metric": "compounded_relative_excess", "value": development["compounded_relative_excess_return"]},
        {"metric": "mean_icir", "value": development["mean_icir"]},
        {"metric": "mean_rank_ic", "value": development["mean_rank_ic"]},
        {"metric": "mean_top_bottom_spread", "value": development["mean_top_bottom_spread"]},
        {"metric": "positive_excess_windows", "value": development["positive_excess_windows"]},
        {"metric": "worst_drawdown", "value": development["worst_drawdown"]},
    ]
)

In [ ]:
pd.DataFrame(development["windows"])

Interpretation:

- 2024H1 and 2024H2 produced positive portfolio excess despite negative Rank IC.
- 2025H1 was the strongest ranking-aligned development window.
- 2025H2 underperformed CSI 300.
- Mean development Rank IC was only 0.0042, so the headline return cannot be treated as clean stock-ranking alpha.

## 5. Frozen 2026H1 evidence

The 2026H1 window was evaluated once in workflow run `30733728747`, artifact `8828889722`. It strongly supported score orientation but did not select a new model.

In [ ]:
challenge = config["backtest_evidence"]["frozen_challenge"]
pd.DataFrame([challenge]).T.rename(columns={0: "value"})

## 6. Provider snapshot sensitivity

The identical XGBoost contract produced different historical economics on two promotion-eligible provider snapshots. This is why a model version and an evidence revision are separate concepts.

In [ ]:
comparison = config["backtest_evidence"]["snapshot_comparison"]
pd.DataFrame(
    [
        {"snapshot": "prior", **comparison["prior_provider"]},
        {"snapshot": "canonical", **comparison["canonical_provider"]},
    ]
)

No further CN factor or parameter search is authorized until Issue #345 explains whether the difference is appended-only behavior, legitimate historical revision, pipeline nondeterminism or unresolved provider drift.

## 7. Evidence identity

In [ ]:
pd.DataFrame([config["evidence_identity"]]).T.rename(columns={0: "value"})

## 8. Reproduction

Contract validation is fast. Full model execution requires the exact governed provider snapshot. Rebuilding a current provider may create a new evidence revision and must not be described as reproducing the canonical snapshot unless its identity matches.

In [ ]:
VALIDATE_COMMAND = [
    sys.executable,
    str(ROOT / "scripts/validate_model_x1_baselines.py"),
]
FULL_BACKTEST_COMMAND = [
    "uv", "run", "python", "scripts/run_cn_feature_quality_validation.py",
    "--spec", "configs/research_paradigms/cn_x1_0_frozen_v1.yaml",
    "--provider-uri", "artifacts/selected_pool_price_refresh/cn/data/providers/cn",
    "--output-dir", "artifacts/evidence/model_versions/cn_x1_0",
]

print("Validate:", " ".join(VALIDATE_COMMAND))
print("Full backtest:", " ".join(FULL_BACKTEST_COMMAND))

RUN_CONTRACT_VALIDATION = False
RUN_FULL_BACKTEST = False

if RUN_CONTRACT_VALIDATION:
    subprocess.run(VALIDATE_COMMAND, cwd=ROOT, check=True)

if RUN_FULL_BACKTEST:
    subprocess.run(FULL_BACKTEST_COMMAND, cwd=ROOT, check=True)

## 9. Known limitations and next version

CN x1.0 is immutable. Effective experiments may propose **CN x1.1**, but only after the provider-drift gate passes.

Required evidence:

1. immutable provider comparison and revision classification;
2. sector, size, beta and volatility exposure decomposition;
3. seed and block-bootstrap stability;
4. ranking/economic alignment across multiple windows;
5. one new untouched challenge window.

In [ ]:
pd.DataFrame({"known_limitation": config["known_limitations"]})